In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import os
import sys
import json
from pathlib import Path
import random
import inspect
from pprint import pprint

from dotenv import load_dotenv, find_dotenv
# 1. Locate and load the environment variables from the .env configuration file
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# 2. Extract configuration variables from the environment
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
REPOS_DIR = os.getenv("REPOS_DIR")
DATA_DIR = os.getenv("DATA_DIR")

project_root = Path(PROJECT_ROOT).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import your newly structured module
from kg_commit.knowledge.dataloader import CommitDataLoader, JITDatasetAdapter
from kg_commit.knowledge.parsers import FilteredCommitParser, IdentityCommitParser
from kg_commit.knowledge.utils import CommitPayloadPrinter
from kg_commit.knowledge.graph import JITCommitKnowledgeGraph

In [8]:
def generate_repo_map(base_dir: str, prefix: str = "apache/") -> dict[str, str]:
    """
    Scans a base directory for valid git repositories and constructs
    a REPO_MAP dictionary compatible with the CommitDataLoader mapping schema.
    """
    repo_map = {}
    base_path = Path(base_dir)
    
    if not base_path.exists():
        print(f"⚠️ Warning: Base directory '{base_dir}' does not exist.")
        return repo_map

    # Iterate through all direct items in the repos folder
    for item in base_path.iterdir():
        if item.is_dir():
            # Check if it contains a hidden .git directory to verify it's a real repo
            git_dir = item / ".git"
            if git_dir.exists():
                # Reconstruct the project key name (e.g., "apache/groovy")
                project_key = f"{prefix}{item.name.lower()}"
                
                # Assign the absolute string path as the value
                repo_map[project_key] = str(item.resolve())
                
    return repo_map

# Run the auto-generation mapping
REPO_MAP = generate_repo_map(REPOS_DIR, prefix="apache/")

# Print out your freshly discovered mappings
print("📂 Automatically generated REPO_MAP mappings:")
print("-" * 50)
for project, local_path in REPO_MAP.items():
    print(f"  '{project}'")
print("-" * 50)

📂 Automatically generated REPO_MAP mappings:
--------------------------------------------------
  'apache/activemq'
  'apache/camel'
  'apache/cassandra'
  'apache/flink'
  'apache/groovy'
  'apache/hadoop'
  'apache/hadoop-hdfs'
  'apache/hadoop-mapreduce'
  'apache/hbase'
  'apache/hive'
  'apache/ignite'
  'apache/kafka'
  'apache/spark'
  'apache/zeppelin'
  'apache/zookeeper'
--------------------------------------------------


In [9]:
# Neo4j Local Instance Credentials
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password1234"

In [11]:
# =====================================================================
# INITIALIZATION
# =====================================================================
data_loader = CommitDataLoader(repo_map=REPO_MAP)

# Instantiate the structural graph engine using our default parser schema
parser_instance = FilteredCommitParser()
kg = JITCommitKnowledgeGraph(
    uri=NEO4J_URI,
    auth_user=NEO4J_USER,
    auth_pass=NEO4J_PASSWORD,
    parser=parser_instance
)

In [19]:
from tqdm import tqdm
import time

# 1. Wipe historical graph database state for a fresh chronological run
print("Wiping historical graph database state...")
counters = kg.purge_database()
print(f"Database wiped successfully. Nodes/Relationships removed: {counters}\n")

Wiping historical graph database state...
Database wiped successfully. Nodes/Relationships removed: SummaryCounters{contains_updates: False, contains_system_updates: False}



In [12]:
from typing import Iterable, Any, Dict, Generator
from tqdm import tqdm

def make_tracked_stream(
    commit_stream: Iterable[Dict[str, Any]], 
    project_label: str
) -> Generator[Dict[str, Any], None, None]:
    """
    Wraps a flat raw commit stream generator with a clean, dynamic tqdm progress bar.
    Tracks throughput and updates terminal UI frames cleanly on-the-fly.
    """
    with tqdm(
        desc=f" [{project_label}] Ingest", 
        unit=" commit", 
        dynamic_ncols=True
    ) as pbar:
        
        for current_count, parsed_commit in enumerate(commit_stream, start=1):
            commit_id = parsed_commit.get("commit_id", "Unknown")
            
            # Update visual diagnostics side-cards
            pbar.set_description(f"Processing {commit_id[:8]}")
            pbar.set_postfix(total_ingested=current_count)
            pbar.update(1)
            
            yield parsed_commit

In [ ]:
# Global portfolio metrics collection definitions
portfolio_ingest_stats = {}
global_pipeline_start = time.time()

print("🚀 COMMENCING PORTFOLIO-WIDE CHRONOLOGICAL INGESTION STREAMS...")
print("=" * 85)

# 2. Sequential execution over discovered repositories
for project_key in REPO_MAP.keys():
    print(f"\n⏳ Activating target stream: '{project_key}'")
    print("-" * 85)
    
    project_start_time = time.time()
    project_short_name = project_key.split('/')[-1]
    
    try:
        # A. Initialize the fast underlying process stream generator
        raw_stream = data_loader.fetch_all_commits_fast(project=project_key, limit=-1)
        
        # B. Intercept the generator stream with our tracking telemetry
        tracked_stream = make_tracked_stream(raw_stream, project_label=project_short_name)
        
        # C. Pipe the tracked stream directly into the reusable Neo4j session engine
        actual_ingested = kg.ingest_fast(tracked_stream)
        
        # D. Calculate throughput performance metrics
        project_duration = time.time() - project_start_time
        throughput = actual_ingested / project_duration if project_duration > 0 else 0
        
        # Save records for the portfolio-wide overview report
        portfolio_ingest_stats[project_key] = {
            "ingested": actual_ingested,
            "duration": project_duration,
            "throughput": throughput
        }
        
        print(f"✅ Finished '{project_key}': Ingested {actual_ingested} commits in {project_duration:.3f}s ({throughput:.2f} commits/sec)")
        print("-" * 85)
        
    except Exception as e:
        print(f"❌ Critical Pipeline Failure on project '{project_key}': {str(e)}")
        portfolio_ingest_stats[project_key] = {"ingested": 0, "duration": 0.0, "throughput": 0.0}
        continue

global_pipeline_duration = time.time() - global_pipeline_start

# =====================================================================
# FINAL PORTFOLIO INGESTION REPORT DASHBOARD
# =====================================================================
print("\n" + "=" * 90)
print("                    PORTFOLIO GRAPH INGESTION ANALYTICS SUMMARY                 ")
print("=" * 90)
print(f"{'PROJECT KEY':<25} | {'INGESTED COMMITS':<18} | {'TIME ELAPSED':<14} | {'INGEST SPEED (C/s)'}")
print("-" * 90)

grand_total_ingested = 0
for proj, metrics in portfolio_ingest_stats.items():
    grand_total_ingested += metrics["ingested"]
    print(f"{proj:<25} | {metrics['ingested']:<18} | {metrics['duration']:11.3f}s | {metrics['throughput']:.2f} commits/sec")

print("-" * 90)
print(f"Portfolio Summary metrics:")
print(f" ├── Total Projects Successfully Processed : {len(portfolio_ingest_stats)}")
print(f" ├── Grand Total Ingested Graph Commits    : {grand_total_ingested}")
print(f" └── Total Pipeline Execution Wall Time     : {global_pipeline_duration:.3f} seconds")
print("=" * 90)

Wiping historical graph database state...
Database wiped successfully. Elements removed: SummaryCounters{nodes_deleted: 32514, relationships_deleted: 99608, contains_updates: True, contains_system_updates: False}

🚀 COMMENCING PORTFOLIO-WIDE CHRONOLOGICAL INGESTION STREAMS...

⏳ Activating target stream: 'apache/activemq'
-------------------------------------------------------------------------------------


Processing 8ce74fa4: : 835 commit [00:57, 24.99 commit/s, total_ingested=836]

In [ ]:
# # Wipe historical state for a fresh chronological run
# print("Wiping historical graph database state...")
# kg.purge_database()

# # =====================================================================
# # DIRECT REPOSITORY LOG STREAMING (CHRONOLOGICAL)
# # =====================================================================
# print(f"\nOpening local repository at: {REPO_PATH}")
# # Access the underlying cached Repo instance inside the loader
# repo = data_loader._get_repo(PROJECT_KEY)

# # Stream all commits from the repository, ordered from past to present
# print("Extracting commit timeline in ascending chronological order (oldest -> newest)...")
# chronological_commits = list(repo.iter_commits(reverse=True))
# total_commits = len(chronological_commits)

# print(f"Discovered a total of {total_commits} historical commits inside the repo history.\n")
# print("=" * 60)

# ingested_count = 0
# skipped_count = 0

# from tqdm import tqdm

# # Wipe historical state for a fresh chronological run
# print("Wiping historical graph database state...")
# kg.purge_database()

# # =====================================================================
# # DIRECT REPOSITORY LOG STREAMING (CHRONOLOGICAL WITH TQDM)
# # =====================================================================
# print(f"\nOpening local repository at: {REPO_PATH}")
# repo = data_loader._get_repo(PROJECT_KEY)

# print("Extracting commit timeline in ascending chronological order (oldest -> newest)...")
# chronological_commits = list(repo.iter_commits(reverse=True))
# total_commits = len(chronological_commits)

# print(f"Discovered a total of {total_commits} historical commits inside the repo history.\n")
# print("=" * 60)

# ingested_count = 0
# skipped_count = 0
# LIMIT = -1

# # 1. Slice target scope up to our execution LIMIT
# target_commits = chronological_commits[:LIMIT]

# # 2. Initialize the tqdm context manager manually targeted to our ingestion ceiling
# with tqdm(total=LIMIT, desc="Pipeline Ingestion", unit="commit") as pbar:
    
#     for idx, commit in enumerate(target_commits, start=1):
#         commit_id = commit.hexsha
        
#         # Extract rich repo analytics context (filters for .java files)
#         raw_payload = data_loader.fetch_commit_data(project=PROJECT_KEY, commit_id=commit_id)
        
#         # Skip if the commit didn't alter any Java source blocks
#         if raw_payload is None:
#             skipped_count += 1
#             # Dynamically update metadata counter on the right side of the progress bar
#             pbar.set_postfix(ingested=ingested_count, skipped_non_java=skipped_count)
#             pbar.update(1)
#             continue
            
#         # Inject directly via our object-oriented rule map engine
#         success = kg.ingest(raw_payload)
        
#         if success:
#             ingested_count += 1
            
#             # Use pbar.set_description to flash the currently processing Commit ID over the bar
#             pbar.set_description(f"Ingested {commit_id[:8]}")
            
#             # Dynamically update progress counters in real-time
#             pbar.set_postfix(ingested=ingested_count, skipped_non_java=skipped_count)
#             pbar.update(1)
#         else:
#             # Handle case where transaction failed or dropped out
#             pbar.write(f"⚠️  Database transaction failed on commit block: {commit_id[:10]}")
#             pbar.update(1)

# print("=" * 60)
# print("Pipeline processing sequence finalized successfully!")

In [ ]:
# =====================================================================
# GRAPH METRICS SUMMARY VERIFICATION
# =====================================================================
print("=" * 60)
print("\n--- Processing Pipeline Summary ---")
print(f"Total Evaluated Repo Commits      : {total_commits}")
print(f"Successfully Vectorized (Java)    : {ingested_count}")
print(f"Skipped (Non-Java/Docs/Configs)   : {skipped_count}")

def check_graph_labels(tx):
    query = "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS total ORDER BY total DESC"
    return [row.data() for row in tx.run(query)]

with kg.driver.session() as session:
    distribution = session.execute_read(check_graph_labels)

print("\n--- Neo4j Local Database Topology Status ---")
if not distribution:
    print("Database is currently empty.")
for node_metric in distribution:
    print(f"Node Label: {node_metric['label']:15} | Node Count: {node_metric['total']}")

# Clean up memory resources and close thread channels safely
kg.close()
print("\nSession completed successfully.")